# Imports

In [1]:
import os
import sys

from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import matplotlib
import pickle
import re
import shutil
import yaml

from concurrent.futures import ThreadPoolExecutor, as_completed
from functools import partial
from itertools import product
from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.pyplot as plt
from multiprocessing import Pool
import numpy as np
import pandas as pd
import seaborn as sns

from tools import load_npy, load_yaml_as_df, load_pkl, exist_metric, exist_stf_metric, inverse_stf_metrics, keep_split, is_full_group, load_metric_from_log, load_yaml_fast

ROOT = str(Path.cwd().parent)

plt.style.use('default')
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
plt.rc('font', family='Arial')
matplotlib.rcParams['mathtext.fontset'] = 'stix'
matplotlib.rcParams['font.size'] = 10

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# Baselines Reproduction

## load data

In [2]:
root = '/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/FreDF/results/baselines'
exp_dirs = [entry.path for entry in os.scandir(root) if entry.is_dir()]

def process_exp_dir(exp_dir):
    runned, setting_dir = exist_metric(exp_dir)
    if not runned:
        return None

    config = load_yaml_fast(os.path.join(setting_dir, 'config.yaml'))
    metric = load_yaml_fast(os.path.join(setting_dir, 'metrics.yaml'))
    result = config | metric
    result['exp_dir'] = exp_dir
    return result

# num_workers = os.cpu_count()
num_workers = 64

df = []
with ThreadPoolExecutor(max_workers=num_workers) as ex:
    futures = [ex.submit(process_exp_dir, exp_dir) for exp_dir in exp_dirs]

    for fut in as_completed(futures):
        r = fut.result()
        if r is not None:
            df.append(r)

df = pd.DataFrame(df)

df.sort_values(by=['model', 'data_id', 'pred_len'], inplace=True)

save_root = '/mnt/tidalfs-bdsz01/usr/panlicheng/workspace/FreDF/stats_ext'
os.makedirs(save_root, exist_ok=True)
df.to_csv(f"{save_root}/baselines_repro.csv", index=False)

df.head(4)

,activation,add_noise,anomaly_ratio,auxi_batch_size,auxi_lambda,auxi_loss,auxi_mode,auxi_type,batch_size,c_out,channel_independence,checkpoints,cutoff_freq_percentage,d_ff,d_layers,d_model,data,data_id,data_path,data_percentage,dec_in,des,deterministic,devices,distil,dropout,e_layers,embed,enc_in,extra_metrics,factor,features,first_order,fix_seed,freq,gpu,inner_lr,inverse,is_training,itr,label_len,learning_rate,leg_degree,log_path,loss,lr_decay,lradj,mask_rate,max_norm,meta_inner_steps,meta_lr,meta_optim_type,meta_type,min_lr,mode,model,model_id,model_per_task,module_first,moving_avg,n_heads,noise_amp,noise_freq_percentage,noise_seed,noise_type,num_kernels,num_tasks,num_workers,offload,optim_type,output_attention,output_log,output_pred,output_vis,overlap_ratio,p_hidden_dims,p_hidden_layers,patience,pct_start,pred_len,pretrain_model_path,rec_lambda,reconstruction_type,report_to,rerun,results,root_path,seasonal_patterns,seq_len,step_size,target,task_name,test_batch_size,test_results,thread,top_k,train_epochs,use_amp,use_gpu,use_multi_gpu,verbose,warmup_steps,weighting_type,mae,mse,rmse,mape,mspe,mre,exp_dir
6,gelu,False,0.25,1024,0.0,MAE,fft,complex,16,321,0,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...,0.06,512,1,512,custom,ECL,electricity.csv,1.0,321,iTransformer,0,"0,1,2,3",True,0.1,3,timeF,321,[],3,M,1,2023,h,0,0.0005,False,1,1,48,0.0005,2,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...,MSE,0.5,type1,0.25,1.0,1,0.0005,sgd,all,0.000001,0,iTransformer,ECL_96_96,0,1,25,8,1,0.05,2023,sin,6,5,10,0,adam,False,False,False,False,0.15,"[128, 128]",2,3,0.2,96,None,1.0,imputation,local,0,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...,/mnt/tidalfs-bdsz01/usr/panlicheng/dataset/ele...,Monthly,96,1,OT,long_term_forecast,1,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...,1,5,10,False,True,False,0,20,softmax,0.241299,0.149780,0.387014,2.524126,658142.3125,2.524126,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...
0,gelu,False,0.25,1024,0.0,MAE,fft,complex,16,321,0,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...,0.06,512,1,512,custom,ECL,electricity.csv,1.0,321,iTransformer,0,"0,1,2,3",True,0.1,3,timeF,321,[],3,M,1,2023,h,0,0.0005,False,1,1,48,0.0005,2,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...,MSE,0.5,type1,0.25,1.0,1,0.0005,sgd,all,0.000001,0,iTransformer,ECL_96_192,0,1,25,8,1,0.05,2023,sin,6,5,10,0,adam,False,False,False,False,0.15,"[128, 128]",2,3,0.2,192,None,1.0,imputation,local,0,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...,/mnt/tidalfs-bdsz01/usr/panlicheng/dataset/ele...,Monthly,96,1,OT,long_term_forecast,1,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...,1,5,10,False,True,False,0,20,softmax,0.259894,0.168726,0.410762,2.691765,682559.3125,2.691765,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...
7,gelu,False,0.25,1024,0.0,MAE,fft,complex,16,321,0,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...,0.06,512,1,512,custom,ECL,electricity.csv,1.0,321,iTransformer,0,"0,1,2,3",True,0.1,3,timeF,321,[],3,M,1,2023,h,0,0.0005,False,1,1,48,0.0005,2,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...,MSE,0.5,type1,0.25,1.0,1,0.0005,sgd,all,0.000001,0,iTransformer,ECL_96_336,0,1,25,8,1,0.05,2023,sin,6,5,10,0,adam,False,False,False,False,0.15,"[128, 128]",2,3,0.2,336,None,1.0,imputation,local,0,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...,/mnt/tidalfs-bdsz01/usr/panlicheng/dataset/ele...,Monthly,96,1,OT,long_term_forecast,1,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...,1,5,10,False,True,False,0,20,softmax,0.272118,0.179544,0.423727,2.720321,609929.7500,2.720321,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...
2,gelu,False,0.25,1024,0.0,MAE,fft,complex,16,321,0,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...,0.06,512,1,512,custom,ECL,electricity.csv,1.0,321,iTransformer,0,"0,1,2,3",True,0.1,3,timeF,321,[],3,M,1,2023,h,0,0.0005,False,1,1,48,0.0005,2,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...,MSE,0.5,type1,0.25,1.0,1,0.0005,sgd,all,0.000001,0,iTransformer,ECL_96_720,0,1,25,8,1,0.05,2023,sin,6,5,10,0,adam,Fals

## analysis

In [6]:
min_mode = 'group'

df2 = df.copy()

columns = ['model', 'data_id', 'learning_rate', 'auxi_loss', 'rec_lambda', 'auxi_lambda']
if min_mode == 'group':
    mse_mean = df2.groupby(columns)['mse'].mean().reset_index()
    idx = mse_mean.groupby(['model', 'data_id'])['mse'].idxmin()
    best_model = mse_mean.loc[idx]
    df2 = df2.merge(best_model[columns], on=columns, how='inner')
elif min_mode == 'each':
    min_mse_idx = df2.groupby(['model', 'data_id', 'pred_len'])['mse'].idxmin()
    df2 = df2.loc[min_mse_idx]
elif min_mode == 'sum':
    # 新增：mse+mae最小
    df2['sum_error'] = df2['mse'] + df2['mae']
    min_sum_idx = df2.groupby(['model', 'data_id', 'pred_len'])['sum_error'].idxmin()
    df2 = df2.loc[min_sum_idx]
    df2 = df2.drop(columns=['sum_error'])

columns = ['model', 'pred_len', 'data_id', 'mse', 'mae', 'learning_rate', 'inner_lr', 'meta_lr', 'auxi_loss', 'rec_lambda', 'auxi_lambda', 'lradj', 'train_epochs', 'patience', 'batch_size', 'auxi_batch_size', 'warmup_steps', 'meta_inner_steps', 'overlap_ratio', 'num_tasks', 'max_norm', 'first_order', 'auxi_mode', 'auxi_type', 'exp_dir']
df2 = df2[columns]

dst_order = ['ETTm1', 'ETTm2', 'ETTh1', 'ETTh2', 'ECL', 'Traffic', 'Weather', 'PEMS03', 'PEMS08']
df2['data_id'] = pd.Categorical(df2['data_id'], categories=dst_order, ordered=True)

model_order = ['iTransformer']
df2['model'] = pd.Categorical(df2['model'], categories=model_order, ordered=True)
df2.sort_values(by=['model', 'data_id', 'pred_len'], inplace=True)

# save_root = '/mnt/tidalfs-bdsz01/usr/panlicheng/workspace/TransDF/stats'
# os.makedirs(save_root, exist_ok=True)
# df2.to_csv(f"{save_root}/finetune_best_meta.csv", index=False)

# print(df2)
df2_avg = df2.groupby(['model', 'data_id']).mean(numeric_only=True).reset_index()
df2_avg['pred_len'] = 'Avg'

df2 = pd.concat([df2, df2_avg]).reset_index(drop=True)

# pl_order = [96, 192, 336, 720, 'Avg']
# df2['pred_len'] = pd.Categorical(df2['pred_len'], categories=pl_order, ordered=True)

df2.sort_values(by=['data_id', 'model', 'pred_len'], inplace=True)
df2.dropna(inplace=True, thresh=5)
df2

df2_show = df2.set_index(['data_id', 'pred_len', 'model']).unstack('model').swaplevel(axis=1)
columns = []
for model in df2_show.columns.levels[0]:
    columns.append((model, 'mse'))
    columns.append((model, 'mae'))
df2_show = df2_show[columns]
df2_show


/tmp/ipykernel_1162988/615453523.py:36: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df2_avg = df2.groupby(['model', 'data_id']).mean(numeric_only=True).reset_index()


model            iTransformer          
                          mse       mae
data_id pred_len                       
ETTm1   96           0.336352  0.372312
        192          0.384524  0.397493
        336          0.423755  0.422011
        720          0.489180  0.458688
        Avg          0.408453  0.412626
ETTm2   96           0.182629  0.265135
        192          0.250751  0.311133
        336          0.315292  0.350480
        720          0.420125  0.410055
        Avg          0.292199  0.334201
ETTh1   96           0.385397  0.405851
        192          0.443168  0.439986
        336          0.479857  0.457184
        720          0.496299  0.486989
        Avg          0.451180  0.447503
ETTh2   96           0.301347  0.349246
        192          0.383043  0.397184
        336          0.424907  0.432255
        720          0.436077  0.449450
        Avg          0.386343  0.407034
ECL     96           0.149780  0.241299
        192          0.168726  0.259894
        336          0.179544  0.272118
        720          0.212531  0.301294
        Avg          0.177645  0.268651
Traffic 96           0.396536  0.271301
        192          0.415468  0.278797
        336          0.430330  0.286088
        720          0.462488  0.303103
        Avg          0.426206  0.284822
Weather 96           0.178926  0.219483
        192          0.222832  0.255949
        336          0.281764  0.299582
        720          0.370373  0.360043
        Avg          0.263474  0.283764
PEMS03  12           0.069979  0.176055
        24           0.100166  0.212336
        36           0.133197  0.247060
        48           0.168572  0.279045
        Avg          0.117979  0.228624
PEMS08  12           0.083028  0.185949
        24           0.123187  0.226983
        36           0.168626  0.266757
        48           0.218459  0.306175
        Avg          0.148325  0.246466

# FreDF Reproduction

## load data

In [7]:
root = '/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/FreDF/results/ltf_repro'
exp_dirs = [entry.path for entry in os.scandir(root) if entry.is_dir()]

def process_exp_dir(exp_dir):
    runned, setting_dir = exist_metric(exp_dir)
    if not runned:
        return None

    config = load_yaml_fast(os.path.join(setting_dir, 'config.yaml'))
    metric = load_yaml_fast(os.path.join(setting_dir, 'metrics.yaml'))
    result = config | metric
    result['exp_dir'] = exp_dir
    return result

# num_workers = os.cpu_count()
num_workers = 64

df = []
with ThreadPoolExecutor(max_workers=num_workers) as ex:
    futures = [ex.submit(process_exp_dir, exp_dir) for exp_dir in exp_dirs]

    for fut in as_completed(futures):
        r = fut.result()
        if r is not None:
            df.append(r)

df = pd.DataFrame(df)

df.sort_values(by=['model', 'data_id', 'pred_len'], inplace=True)

save_root = '/mnt/tidalfs-bdsz01/usr/panlicheng/workspace/FreDF/stats_ext'
os.makedirs(save_root, exist_ok=True)
df.to_csv(f"{save_root}/ltf_repro.csv", index=False)

df.head(4)

,activation,add_noise,anomaly_ratio,auxi_batch_size,auxi_lambda,auxi_loss,auxi_mode,auxi_type,batch_size,c_out,channel_independence,checkpoints,cutoff_freq_percentage,d_ff,d_layers,d_model,data,data_id,data_path,data_percentage,dec_in,des,deterministic,devices,distil,dropout,e_layers,embed,enc_in,extra_metrics,factor,features,first_order,fix_seed,freq,gpu,inner_lr,inverse,is_training,itr,label_len,learning_rate,leg_degree,log_path,loss,lr_decay,lradj,mask_rate,max_norm,meta_inner_steps,meta_lr,meta_optim_type,meta_type,min_lr,mode,model,model_id,model_per_task,module_first,moving_avg,n_heads,noise_amp,noise_freq_percentage,noise_seed,noise_type,num_kernels,num_tasks,num_workers,offload,optim_type,output_attention,output_log,output_pred,output_vis,overlap_ratio,p_hidden_dims,p_hidden_layers,patience,pct_start,pred_len,pretrain_model_path,rec_lambda,reconstruction_type,report_to,rerun,results,root_path,seasonal_patterns,seq_len,step_size,target,task_name,test_batch_size,test_results,thread,top_k,train_epochs,use_amp,use_gpu,use_multi_gpu,verbose,warmup_steps,weighting_type,mae,mse,rmse,mape,mspe,mre,exp_dir
2,gelu,False,0.25,1024,0.9,MAE,rfft,complex,16,321,0,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...,0.06,512,1,512,custom,ECL,electricity.csv,1.0,321,iTransformer,0,"0,1,2,3",True,0.1,3,timeF,321,[],3,M,1,2023,h,0,0.0005,False,1,1,48,0.001,2,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...,MSE,0.5,type1,0.25,1.0,1,0.0005,sgd,all,0.000001,0,iTransformer,ECL_96_96,0,1,25,8,1,0.05,2023,sin,6,5,10,0,adam,False,False,False,False,0.15,"[128, 128]",2,3,0.2,96,None,0.1,imputation,local,0,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...,/mnt/tidalfs-bdsz01/usr/panlicheng/dataset/ele...,Monthly,96,1,OT,long_term_forecast,1,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...,1,5,10,False,True,False,0,20,softmax,0.234586,0.145751,0.381773,2.475816,566563.18750,2.475816,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...
1,gelu,False,0.25,1024,0.9,MAE,rfft,complex,16,321,0,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...,0.06,512,1,512,custom,ECL,electricity.csv,1.0,321,iTransformer,0,"0,1,2,3",True,0.1,3,timeF,321,[],3,M,1,2023,h,0,0.0005,False,1,1,48,0.001,2,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...,MSE,0.5,type1,0.25,1.0,1,0.0005,sgd,all,0.000001,0,iTransformer,ECL_96_192,0,1,25,8,1,0.05,2023,sin,6,5,10,0,adam,False,False,False,False,0.15,"[128, 128]",2,3,0.2,192,None,0.1,imputation,local,0,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...,/mnt/tidalfs-bdsz01/usr/panlicheng/dataset/ele...,Monthly,96,1,OT,long_term_forecast,1,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...,1,5,10,False,True,False,0,20,softmax,0.248186,0.159943,0.399929,2.609770,630721.37500,2.609770,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...
0,gelu,False,0.25,1024,0.9,MAE,rfft,complex,16,321,0,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...,0.06,512,1,512,custom,ECL,electricity.csv,1.0,321,iTransformer,0,"0,1,2,3",True,0.1,3,timeF,321,[],3,M,1,2023,h,0,0.0005,False,1,1,48,0.001,2,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...,MSE,0.5,type1,0.25,1.0,1,0.0005,sgd,all,0.000001,0,iTransformer,ECL_96_336,0,1,25,8,1,0.05,2023,sin,6,5,10,0,adam,False,False,False,False,0.15,"[128, 128]",2,3,0.2,336,None,0.1,imputation,local,0,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...,/mnt/tidalfs-bdsz01/usr/panlicheng/dataset/ele...,Monthly,96,1,OT,long_term_forecast,1,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...,1,5,10,False,True,False,0,20,softmax,0.264276,0.173250,0.416234,2.784483,743406.12500,2.784483,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...
3,gelu,False,0.25,1024,0.9,MAE,rfft,complex,16,321,0,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...,0.06,512,1,512,custom,ECL,electricity.csv,1.0,321,iTransformer,0,"0,1,2,3",True,0.1,3,timeF,321,[],3,M,1,2023,h,0,0.0005,False,1,1,48,0.001,2,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...,MSE,0.5,type1,0.25,1.0,1,0.0005,sgd,all,0.000001,0,iTransformer,ECL_96_720,0,1,25,8,1,0.05,2023,sin,6,5,10,0,adam,F

## analysis

In [9]:
min_mode = 'each'

df2 = df.copy()

columns = ['model', 'data_id', 'learning_rate', 'auxi_loss', 'rec_lambda', 'auxi_lambda']
if min_mode == 'group':
    mse_mean = df2.groupby(columns)['mse'].mean().reset_index()
    idx = mse_mean.groupby(['model', 'data_id'])['mse'].idxmin()
    best_model = mse_mean.loc[idx]
    df2 = df2.merge(best_model[columns], on=columns, how='inner')
elif min_mode == 'each':
    min_mse_idx = df2.groupby(['model', 'data_id', 'pred_len'])['mse'].idxmin()
    df2 = df2.loc[min_mse_idx]
elif min_mode == 'sum':
    # 新增：mse+mae最小
    df2['sum_error'] = df2['mse'] + df2['mae']
    min_sum_idx = df2.groupby(['model', 'data_id', 'pred_len'])['sum_error'].idxmin()
    df2 = df2.loc[min_sum_idx]
    df2 = df2.drop(columns=['sum_error'])

columns = ['model', 'pred_len', 'data_id', 'mse', 'mae', 'learning_rate', 'auxi_loss', 'rec_lambda', 'auxi_lambda', 'lradj', 'train_epochs', 'patience', 'batch_size', 'auxi_mode', 'auxi_type', 'module_first', 'exp_dir']
df2 = df2[columns]

dst_order = ['ETTm1', 'ETTm2', 'ETTh1', 'ETTh2', 'ECL', 'Traffic', 'Weather', 'PEMS03', 'PEMS08']
df2['data_id'] = pd.Categorical(df2['data_id'], categories=dst_order, ordered=True)

model_order = ['iTransformer']
df2['model'] = pd.Categorical(df2['model'], categories=model_order, ordered=True)
df2.sort_values(by=['model', 'data_id', 'pred_len'], inplace=True)

# save_root = '/mnt/tidalfs-bdsz01/usr/panlicheng/workspace/TransDF/stats'
# os.makedirs(save_root, exist_ok=True)
# df2.to_csv(f"{save_root}/finetune_best_meta.csv", index=False)

# print(df2)
df2_avg = df2.groupby(['model', 'data_id']).mean(numeric_only=True).reset_index()
df2_avg['pred_len'] = 'Avg'

df2 = pd.concat([df2, df2_avg]).reset_index(drop=True)

# pl_order = [96, 192, 336, 720, 'Avg']
# df2['pred_len'] = pd.Categorical(df2['pred_len'], categories=pl_order, ordered=True)

df2.sort_values(by=['data_id', 'model', 'pred_len'], inplace=True)
df2.dropna(inplace=True, thresh=5)
df2

df2_show = df2.set_index(['data_id', 'pred_len', 'model']).unstack('model').swaplevel(axis=1)
columns = []
for model in df2_show.columns.levels[0]:
    columns.append((model, 'mse'))
    columns.append((model, 'mae'))
df2_show = df2_show[columns]
df2_show


/tmp/ipykernel_1162988/3115361632.py:36: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df2_avg = df2.groupby(['model', 'data_id']).mean(numeric_only=True).reset_index()


model            iTransformer          
                          mse       mae
data_id pred_len                       
ETTm1   96           0.324339  0.361389
        192          0.375584  0.386760
        336          0.405854  0.407125
        720          0.471711  0.443577
        Avg          0.394372  0.399713
ETTm2   96           0.173799  0.253205
        192          0.238867  0.296337
        336          0.299498  0.335135
        720          0.399801  0.393866
        Avg          0.277991  0.319636
ETTh1   96           0.377563  0.397369
        192          0.430417  0.426127
        336          0.465102  0.442806
        720          0.471713  0.468807
        Avg          0.436199  0.433778
ETTh2   96           0.286870  0.336230
        192          0.364558  0.385735
        336          0.419588  0.427119
        720          0.421397  0.440241
        Avg          0.373103  0.397331
ECL     96           0.145751  0.234586
        192          0.159943  0.248186
        336          0.173250  0.264276
        720          0.215843  0.299124
        Avg          0.173697  0.261543
Traffic 96           0.394304  0.267797
        192          0.413250  0.274605
        336          0.428792  0.282185
        720          0.464176  0.300701
        Avg          0.425131  0.281322
Weather 96           0.168278  0.204955
        192          0.219921  0.253644
        336          0.280456  0.296220
        720          0.375010  0.361974
        Avg          0.260916  0.279198
PEMS03  12           0.069527  0.174440
        24           0.099326  0.209045
        36           0.134622  0.246612
        48           0.173931  0.282786
        Avg          0.119351  0.228221
PEMS08  12           0.082609  0.185026
        24           0.123446  0.225971
        36           0.168589  0.265980
        48           0.215420  0.301960
        Avg          0.147516  0.244734

# FreDF Extension

## load data

In [2]:
root = '/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/FreDF/results/long_term_ext'
exp_dirs = [entry.path for entry in os.scandir(root) if entry.is_dir()]

def process_exp_dir(exp_dir):
    runned, setting_dir = exist_metric(exp_dir)
    if not runned:
        return None

    config = load_yaml_fast(os.path.join(setting_dir, 'config.yaml'))
    metric = load_yaml_fast(os.path.join(setting_dir, 'metrics.yaml'))
    result = config | metric
    result['exp_dir'] = exp_dir
    return result

# num_workers = os.cpu_count()
num_workers = 64

df = []
with ThreadPoolExecutor(max_workers=num_workers) as ex:
    futures = [ex.submit(process_exp_dir, exp_dir) for exp_dir in exp_dirs]

    for fut in as_completed(futures):
        r = fut.result()
        if r is not None:
            df.append(r)

df = pd.DataFrame(df)

df.sort_values(by=['model', 'data_id', 'pred_len'], inplace=True)

save_root = '/mnt/tidalfs-bdsz01/usr/panlicheng/workspace/FreDF/stats_ext'
os.makedirs(save_root, exist_ok=True)
df.to_csv(f"{save_root}/ltf_meta.csv", index=False)

df.head(4)

,activation,add_noise,anomaly_ratio,auxi_batch_size,auxi_lambda,auxi_loss,auxi_mode,auxi_type,batch_size,c_out,channel_independence,checkpoints,cutoff_freq_percentage,d_ff,d_layers,d_model,data,data_id,data_path,data_percentage,dec_in,des,deterministic,devices,distil,dropout,e_layers,embed,enc_in,extra_metrics,factor,features,first_order,fix_seed,freq,gpu,inner_lr,inverse,is_training,itr,label_len,learning_rate,leg_degree,log_path,loss,lr_decay,lradj,mask_rate,max_norm,meta_inner_steps,meta_lr,meta_optim_type,min_lr,mode,model,model_id,model_per_task,module_first,moving_avg,n_heads,noise_amp,noise_freq_percentage,noise_seed,noise_type,num_kernels,num_tasks,num_workers,offload,optim_type,output_attention,output_log,output_pred,output_vis,overlap_ratio,p_hidden_dims,p_hidden_layers,patience,pct_start,pred_len,pretrain_model_path,rec_lambda,reconstruction_type,report_to,rerun,results,root_path,seasonal_patterns,seq_len,step_size,target,task_name,test_batch_size,test_results,thread,top_k,train_epochs,use_amp,use_gpu,use_multi_gpu,verbose,warmup_steps,mae,mse,rmse,mape,mspe,mre,loss_total,loss_feq,loss_rec,loss_auxi,exp_dir
245,gelu,False,0.25,64,0.9,MAE,rfft,complex,16,321,0,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...,0.06,512,1,512,custom,ECL,electricity.csv,1.0,321,iTransformer,0,"0,1,2,3",True,0.1,3,timeF,321,[],3,M,1,2023,h,0,0.0100,False,1,1,48,0.0100,2,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...,MSE,0.5,type1,0.25,5.0,2,0.02,sgd,0.000001,0,iTransformer,ECL_96_96,0,1,25,8,1,0.05,2023,sin,6,3,10,0,adam,False,False,False,False,0.0,"[128, 128]",2,5,0.2,96,None,0.1,imputation,local,0,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...,/mnt/tidalfs-bdsz01/usr/panlicheng/dataset/ele...,Monthly,96,1,OT,long_term_forecast_meta_ml3,1,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...,1,5,30,False,True,False,0,300,0.762022,0.845924,0.919741,4.038494,1.147098e+06,4.038494,2.604919,3.882636,0.845924,2.800363,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...
262,gelu,False,0.25,64,0.9,MAE,rfft,complex,16,321,0,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...,0.06,512,1,512,custom,ECL,electricity.csv,1.0,321,iTransformer,0,"0,1,2,3",True,0.1,3,timeF,321,[],3,M,1,2023,h,0,0.0050,False,1,1,48,0.0050,2,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...,MSE,0.5,type1,0.25,5.0,2,0.10,sgd,0.000001,0,iTransformer,ECL_96_96,0,1,25,8,1,0.05,2023,sin,6,3,10,0,adam,False,False,False,False,0.0,"[128, 128]",2,5,0.2,96,None,0.1,imputation,local,0,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...,/mnt/tidalfs-bdsz01/usr/panlicheng/dataset/ele...,Monthly,96,1,OT,long_term_forecast_meta_ml3,1,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...,1,5,30,False,True,False,0,300,0.557620,0.535044,0.731467,4.373426,2.457928e+06,4.373426,1.914490,3.506793,0.535044,2.067762,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...
267,gelu,False,0.25,64,0.9,MAE,rfft,complex,16,321,0,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...,0.06,512,1,512,custom,ECL,electricity.csv,1.0,321,iTransformer,0,"0,1,2,3",True,0.1,3,timeF,321,[],3,M,1,2023,h,0,0.0010,False,1,1,48,0.0010,2,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...,MSE,0.5,type1,0.25,5.0,2,0.01,sgd,0.000001,0,iTransformer,ECL_96_96,0,1,25,8,1,0.05,2023,sin,6,2,10,0,adam,False,False,False,False,0.0,"[128, 128]",2,5,0.2,96,None,0.1,imputation,local,0,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...,/mnt/tidalfs-bdsz01/usr/panlicheng/dataset/ele...,Monthly,96,1,OT,long_term_forecast_meta_ml3,1,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...,1,5,30,False,True,False,0,300,0.234918,0.146056,0.382172,2.479720,5.796439e+05,2.479720,1.828826,2.183565,0.146056,2.015801,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...
274,gelu,False,0.25,64,0.4,MAE,rfft,complex,16,321,0,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...,0.06,512,1,512,custom,ECL,electricity.csv,1.0,321,iTransformer,0,"0,1,2,3",True,0.1,3,timeF,321,[],3,M,1,2023,h,0,0.0008,False,1,1,48,0.0008,2,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...,MSE,0.5,

## analysis

In [12]:
min_mode = 'sum'

df2 = df.copy()

columns = ['model', 'data_id', 'learning_rate', 'auxi_loss', 'rec_lambda', 'auxi_lambda']
if min_mode == 'group':
    mse_mean = df2.groupby(columns)['mse'].mean().reset_index()
    idx = mse_mean.groupby(['model', 'data_id'])['mse'].idxmin()
    best_model = mse_mean.loc[idx]
    df2 = df2.merge(best_model[columns], on=columns, how='inner')
elif min_mode == 'each':
    min_mse_idx = df2.groupby(['model', 'data_id', 'pred_len'])['mse'].idxmin()
    df2 = df2.loc[min_mse_idx]
elif min_mode == 'sum':
    # 新增：mse+mae最小
    df2['sum_error'] = df2['mse'] + df2['mae']
    min_sum_idx = df2.groupby(['model', 'data_id', 'pred_len'])['sum_error'].idxmin()
    df2 = df2.loc[min_sum_idx]
    df2 = df2.drop(columns=['sum_error'])

columns = ['model', 'pred_len', 'data_id', 'mse', 'mae', 'learning_rate', 'rec_lambda', 'auxi_lambda', 'batch_size', 'inner_lr', 'meta_lr', 'meta_inner_steps', 'num_tasks', 'max_norm', 'warmup_steps', 'auxi_batch_size', 'auxi_mode', 'auxi_type', 'module_first', 'auxi_loss', 'first_order', 'overlap_ratio', 'lradj', 'train_epochs', 'patience', 'exp_dir']
df2 = df2[columns]

dst_order = ['ETTm1', 'ETTm2', 'ETTh1', 'ETTh2', 'ECL', 'Traffic', 'Weather', 'PEMS03', 'PEMS08']
df2['data_id'] = pd.Categorical(df2['data_id'], categories=dst_order, ordered=True)

model_order = ['iTransformer']
df2['model'] = pd.Categorical(df2['model'], categories=model_order, ordered=True)
df2.sort_values(by=['model', 'data_id', 'pred_len'], inplace=True)

# save_root = '/mnt/tidalfs-bdsz01/usr/panlicheng/workspace/TransDF/stats'
# os.makedirs(save_root, exist_ok=True)
# df2.to_csv(f"{save_root}/finetune_best_meta.csv", index=False)

# print(df2)
df2_avg = df2.groupby(['model', 'data_id']).mean(numeric_only=True).reset_index()
df2_avg['pred_len'] = 'Avg'

df2 = pd.concat([df2, df2_avg]).reset_index(drop=True)

df2.sort_values(by=['data_id', 'model', 'pred_len'], inplace=True)
df2.dropna(inplace=True, thresh=5)
df2

df2_show = df2.set_index(['data_id', 'pred_len', 'model']).unstack('model').swaplevel(axis=1)
columns = []
for model in df2_show.columns.levels[0]:
    columns.append((model, 'mse'))
    columns.append((model, 'mae'))
df2_show = df2_show[columns]
df2_show


/tmp/ipykernel_1162988/399444542.py:36: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df2_avg = df2.groupby(['model', 'data_id']).mean(numeric_only=True).reset_index()


model            iTransformer          
                          mse       mae
data_id pred_len                       
ETTm1   96           0.320135  0.357688
        192          0.363081  0.381573
        336          0.396234  0.403602
        720          0.458687  0.439293
        Avg          0.384534  0.395539
ETTm2   96           0.171669  0.251341
        192          0.236344  0.296502
        336          0.294357  0.332055
        720          0.388367  0.387036
        Avg          0.272684  0.316733
ETTh1   96           0.373342  0.392018
        192          0.421701  0.419684
        336          0.460807  0.439518
        720          0.462591  0.461021
        Avg          0.429610  0.428060
ETTh2   96           0.284730  0.334576
        192          0.356401  0.380000
        336          0.411939  0.423330
        720          0.416514  0.438936
        Avg          0.367396  0.394210
ECL     96           0.142305  0.233029
        192          0.158183  0.247139
        336          0.171086  0.264279
        720          0.202419  0.292394
        Avg          0.168498  0.259210
Traffic 96           0.382514  0.255764
        192          0.399801  0.266584
        336          0.405709  0.272434
        720          0.426100  0.283789
        Avg          0.403531  0.269643
Weather 96           0.173185  0.211141
        192          0.222602  0.256105
        336          0.279075  0.297653
        720          0.357896  0.348119
        Avg          0.258189  0.278254
PEMS03  12           0.066391  0.170390
        24           0.091763  0.200714
        36           0.119346  0.228728
        48           0.146317  0.254551
        Avg          0.105954  0.213596
PEMS08  12           0.075669  0.172733
        24           0.108354  0.203425
        36           0.139684  0.231530
        48           0.173292  0.256372
        Avg          0.124250  0.216015

## output directory

In [5]:
df2.query("model=='iTransformer' and data_id=='ETTh1' and pred_len==96")[['exp_dir']].iloc[0,0]

'/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/FreDF/results/long_term_ext/iTransformer_ETTh1_96_0.0_1.0_0.002_type1_30_5_32_rfft_complex_MAE_1_1_0.0_64_5.0_3_5_300_0.002_0.1'

## load events

In [6]:
import torch

exp_dir = '/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/FreDF/results/long_term_ext/iTransformer_ETTh1_96_0.0_1.0_0.002_type1_30_5_32_rfft_complex_MAE_1_1_0.0_64_5.0_3_5_300_0.002_0.1'
runned, setting_dir = exist_metric(exp_dir)
events = torch.load(os.path.join(setting_dir, 'events.pth'))
events['scalars'].keys()

/tmp/ipykernel_1976061/4184351160.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  events = torch.load(os.path.join(setting_dir, 'events.pth'))


dict_keys(['96/meta_train/meta_lr', '96/meta_train/task_1_meta_loss', '96/meta_train/task_1_meta_loss_rec', '96/meta_train/task_1_meta_loss_auxi', '96/meta_train/task_2_meta_loss', '96/meta_train/task_2_meta_loss_rec', '96/meta_train/task_2_meta_loss_auxi', '96/meta_train/task_3_meta_loss', '96/meta_train/task_3_meta_loss_rec', '96/meta_train/task_3_meta_loss_auxi', '96/meta_train/meta_loss', '96/meta_train/meta_loss_rec', '96/meta_train/meta_loss_auxi', '96/meta_test/lr', '96/meta_test_iter/loss', '96/meta_test_iter/loss_feq', '96/meta_test_iter/loss_rec', '96/meta_test_iter/loss_auxi', '96/meta_test/loss', '96/meta_test/loss_feq', '96/meta_test/loss_rec', '96/meta_test/loss_auxi', '96/vali/loss', '96/vali/loss_feq', '96/vali/loss_rec', '96/vali/loss_auxi', '96/test/mae', '96/test/mse', '96/test/rmse', '96/test/mape', '96/test/mspe', '96/test/mre', '96/test/loss_total', '96/test/loss_feq', '96/test/loss_rec', '96/test/loss_auxi'])

# report to latex

## load and merge data

In [ ]:
save_root = '/mnt/tidalfs-bdsz01/usr/panlicheng/workspace/ProjDF-Meta/logs'
dfb = pd.read_csv(f"{save_root}/baselines_chosen.csv")

min_mode = 'each'
columns = ['model', 'data_id', 'learning_rate', 'batch_size', 'patience', 'individual', 'train_epochs']
if min_mode == 'group':
    mse_mean = dfb.groupby(columns)['mse'].mean().reset_index()
    idx = mse_mean.groupby(['model', 'data_id'])['mse'].idxmin()
    best_model = mse_mean.loc[idx]
    dfb = dfb.merge(best_model[columns], on=columns, how='inner')
elif min_mode == 'each':
    min_mse_idx = dfb.groupby(['model', 'data_id', 'pred_len'])['mse'].idxmin()
    dfb = dfb.loc[min_mse_idx]

dfb = dfb[['model', 'pred_len', 'data_id', 'mse', 'mae']]
datasets = ['ETTm1', 'ETTm2', 'ETTh1', 'ETTh2', 'ECL', 'Weather', 'PEMS03', 'PEMS08']
dfb = dfb[dfb['data_id'].isin(datasets)]


save_root = '/mnt/tidalfs-bdsz01/usr/panlicheng/workspace/ProjDF-Meta/stats_ML3'
dff = pd.read_csv(f'{save_root}/finetune_best.csv')

dff = dff[
    ((dff.data_id == 'ETTm1') & (dff.model == 'TQNet')) |
    ((dff.data_id == 'ETTm2') & (dff.model == 'TQNet')) |
    ((dff.data_id == 'ETTh1') & (dff.model == 'TQNet')) |
    ((dff.data_id == 'ETTh2') & (dff.model == 'TQNet')) |
    ((dff.data_id == 'ECL') & (dff.model == 'TQNet')) |
    # ((dff.data_id == 'Traffic') & (dff.model == 'TQNet')) |
    ((dff.data_id == 'Weather') & (dff.model == 'TQNet')) |
    ((dff.data_id == 'PEMS03') & (dff.model == 'TQNet')) |
    ((dff.data_id == 'PEMS08') & (dff.model == 'TQNet'))
]

dff['model'] = 'QDF'
dff.to_csv(f"{save_root}/long_term.csv", index=False)
dff = dff[['model', 'pred_len', 'data_id', 'mse', 'mae']]

dft = pd.concat([dfb, dff], axis=0)

# dst_order = ['ETTm1', 'ETTm2', 'ETTh1', 'ETTh2', 'ECL', 'Traffic', 'Weather', 'PEMS03', 'PEMS08']
dst_order = ['ETTm1', 'ETTm2', 'ETTh1', 'ETTh2', 'ECL', 'Weather', 'PEMS03', 'PEMS08']
dft['data_id'] = pd.Categorical(dft['data_id'], categories=dst_order, ordered=True)

model_order = ['QDF', 'TQNet', 'PDF', 'Fredformer', 'iTransformer', 'FreTS', 'TimesNet', 'MICN', 'TiDE', 'PatchTST', 'DLinear']
dft = dft[dft['model'].isin(model_order)]
dft['model'] = pd.Categorical(dft['model'], categories=model_order, ordered=True)

dft_avg = dft.groupby(['model', 'data_id']).mean(numeric_only=True).reset_index()
dft_avg['pred_len'] = 'Avg'

dft = pd.concat([dft, dft_avg]).reset_index(drop=True)
dft.sort_values(by=['model', 'data_id', 'pred_len'], inplace=True)

dft_show = dft.set_index(['data_id', 'pred_len', 'model']).unstack('model').swaplevel(axis=1)
columns = []
for model in dft_show.columns.levels[0]:
    columns.append((model, 'mse'))
    columns.append((model, 'mae'))
dft_show = dft_show[columns]
dft_show

/tmp/ipykernel_1486806/2195932971.py:49: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  dft_avg = dft.groupby(['model', 'data_id']).mean(numeric_only=True).reset_index()


model                  QDF               TQNet                 PDF            \
                       mse       mae       mse       mae       mse       mae   
data_id pred_len                                                               
ETTm1   96        0.306602  0.348971  0.310382  0.351721  0.326234  0.363356   
        192       0.352414  0.376268  0.356100  0.377443  0.364817  0.381136   
        336       0.382595  0.397512  0.387569  0.399900  0.396639  0.402112   
        720       0.441163  0.434478  0.450021  0.436546  0.458370  0.437322   
        Avg       0.370693  0.389307  0.376018  0.391403  0.386515  0.395981   
ETTm2   96        0.170291  0.252600  0.174746  0.255768  0.176168  0.263647   
        192       0.233855  0.293836  0.242704  0.300489  0.245158  0.310199   
        336       0.290130  0.331435  0.297452  0.336435  0.305491  0.345152   
        720       0.386822  0.388532  0.394070  0.393085  0.403959  0.403241   
        Avg       0.270274  0.316601  0.277243  0.321444  0.282694  0.330560   
ETTh1   96        0.365111  0.388897  0.372007  0.391440  0.388068  0.400122   
        192       0.427475  0.421499  0.430498  0.424173  0.440350  0.428146   
        336       0.465918  0.448515  0.486262  0.454420  0.483253  0.448762   
        720       0.466485  0.466534  0.506832  0.485931  0.495008  0.481555   
        Avg       0.431247  0.431361  0.448900  0.438991  0.451670  0.439646   
ETTh2   96        0.285880  0.337922  0.293373  0.342665  0.290784  0.340032   
        192       0.361035  0.388219  0.363695  0.390391  0.374348  0.390743   
        336       0.407637  0.422494  0.411333  0.424371  0.413972  0.425852   
        720       0.419220  0.438823  0.429976  0.443821  0.421089  0.439893   
        Avg       0.368443  0.396864  0.374594  0.400312  0.375048  0.399130   
ECL     96        0.134708  0.228838  0.143456  0.237354  0.175417  0.259347   
        192       0.153000  0.244973  0.160641  0.252081  0.181722  0.265927   
        336       0.169068  0.262404  0.178238  0.269703  0.197201  0.281979   
        720       0.201599  0.290306  0.217629  0.302696  0.237404  0.315331   
        Avg       0.164594  0.256630  0.174991  0.265458  0.197936  0.280646   
Weather 96        0.158300  0.200693  0.159965  0.202723  0.181140  0.221251   
        192       0.206637  0.244936  0.209872  0.247166  0.231525  0.262294   
        336       0.262811  0.286290  0.266696  0.288953  0.285225  0.299993   
        720       0.342250  0.339154  0.346121  0.342470  0.360232  0.348378   
        Avg       0.242499  0.267768  0.245663  0.270328  0.264530  0.282979   
PEMS03  12        0.064309  0.167092  0.097305  0.179638  0.091645  0.204271   
        24        0.080374  0.188680  0.098929  0.204429  0.148696  0.260927   
        36        0.097987  0.208369  0.123181  0.229999  0.210095  0.313705   
        48        0.111981  0.223476  0.157448  0.255610  0.274832  0.364260   
        Avg       0.088663  0.196904  0.119216  0.217419  0.181317  0.285790   
PEMS08  12        0.074487  0.175849  0.078706  0.182539  0.099847  0.208661   
        24        0.104101  0.207774  0.117204  0.222451  0.167697  0.272787   
        36        0.134275  0.236954  0.157844  0.260074  0.244455  0.333303   
        48        0.168359  0.263294  0.203122  0.295390  0.327415  0.388973   
        Avg       0.120305  0.220968  0.139219  0.240114  0.209853  0.300931   

model            Fredformer           iTransformer               FreTS  \
                        mse       mae          mse       mae       mse   
data_id pred_len                                                         
ETTm1   96         0.326369  0.360869     0.337877  0.372283  0.341839   
        192        0.365194  0.382132     0.381666  0.396060  0.384536   
        336        0.395987  0.404369     0.426993  0.423962  0.415986   
        720        0.459217  0.444342     0.495750  0.462558  0.513302   
        Avg        0.386692  0.397928    

## save to latex table

In [ ]:
save_root = "/mnt/tidalfs-bdsz01/usr/panlicheng/workspace/ProjDF-Meta/stats_ML3"
# dft_show.to_latex(f"{save_root}/long_term.tex")



# dst_order = ['ETTm1', 'ETTm2', 'ETTh1', 'ETTh2', 'ECL', 'Traffic', 'Weather', 'PEMS03', 'PEMS08']
dst_order = ['ETTm1', 'ETTm2', 'ETTh1', 'ETTh2', 'ECL', 'Weather', 'PEMS03', 'PEMS08']
model_order = ['QDF', 'TQNet', 'PDF', 'Fredformer', 'iTransformer', 'FreTS', 'TimesNet', 'MICN', 'TiDE', 'PatchTST', 'DLinear']
# 初始化统计字典：每个模型的MSE第1次数和MAE第1次数
mse_count = {model: 0 for model in model_order}
mae_count = {model: 0 for model in model_order}


contents = []
stats_contents = []  # 存储统计行内容
for dst in dst_order:
    dfi = dft[dft.data_id == dst]

    dfi['mse_rank'] = dfi.groupby('pred_len')['mse'].rank(ascending=True, method='dense')
    dfi['mae_rank'] = dfi.groupby('pred_len')['mae'].rank(ascending=True, method='dense')

    # MSE第1名：mse_rank=1的模型
    mse_first_models = dfi[dfi['mse_rank'] == 1]['model']
    for model in mse_first_models:
        if model in mse_count:  # 确保模型在model_order中
            mse_count[model] += 1
    
    # MAE第1名：mae_rank=1的模型
    mae_first_models = dfi[dfi['mae_rank'] == 1]['model']
    for model in mae_first_models:
        if model in mae_count:
            mae_count[model] += 1

    for metric in ['mse', 'mae']:
        dfi[f'{metric}_3f'] = dfi[metric].apply(lambda x: "{:.3f}".format(x))
        # 按排名标记：第1名→\bst{}，第2名→\subbst{}，其他→原格式
        conditions = [dfi[f'{metric}_rank'] == 1, dfi[f'{metric}_rank'] == 2]
        choices = [r'\bst{' + dfi[f'{metric}_3f'] + '}', r'\subbst{' + dfi[f'{metric}_3f'] + '}']
        dfi[f'{metric}_processed'] = np.select(conditions, choices, default=dfi[f'{metric}_3f'])
    dfi = dfi[['model', 'pred_len', 'mse_processed', 'mae_processed']]
    dfi.columns = ['model', 'pred_len', 'mse', 'mae']
    dfi['mse'] = dfi['mse'].apply(lambda x: r"\scalea{" + x + "}")  # 应用\scalea
    dfi['mae'] = dfi['mae'].apply(lambda x: r"\scalea{" + x + "}")
    for i, (pl, group) in enumerate(dfi.groupby('pred_len')):
        if i == 0:
            title = r'\multirow{5}{*}{{\rotatebox{90}{\scalebox{0.95}{' + dst + '}}}}'
            contents.append(title)
        line = f"& {pl} "
        for j, row in enumerate(group.itertuples(index=False)):
            line += f"& {row.mse} & {row.mae}"
        line += " \\\\"
        contents.append(line)
        if i == 3:
            contents.append(r"\cmidrule(lr){2-24}")
        elif i  == 4:
            contents.append("\\midrule\n")

stats_title = r"\multicolumn{2}{c|}{\scalea{{$1^{\text{st}}$ Count}}}"  # 标题：合并前两列
stats_values = []
for model in model_order:
    mse = mse_count[model]
    mae = mae_count[model]
    # 次数>0时用\bst{}标记，否则0
    mse_str = r"\scalea{\bst{" + str(mse) + "}}" if mse > 0 else r"\scalea{" + str(mse) + "}"
    mae_str = r"\scalea{\bst{" + str(mae) + "}}" if mae > 0 else r"\scalea{" + str(mae) + "}"
    stats_values.append(f"{mse_str} & {mae_str}")  # 每个模型的MSE/MAE次数
# 合并统计行
stats_line = f"{stats_title} & {' & '.join(stats_values)} \\\\"
stats_contents.append(stats_line)

print(sum(mse_count.values()), sum(mae_count.values()))
print(max(mse_count.values()), max(mae_count.values()))

with open(f"{save_root}/long_term.tex", "w") as f:
    f.write("\n".join(contents + stats_contents))

40 40
39 39


/tmp/ipykernel_1486806/975220537.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dfi['mse_rank'] = dfi.groupby('pred_len')['mse'].rank(ascending=True, method='dense')
/tmp/ipykernel_1486806/975220537.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dfi['mae_rank'] = dfi.groupby('pred_len')['mae'].rank(ascending=True, method='dense')
/tmp/ipykernel_1486806/975220537.py:35: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_index

## write avg

In [ ]:
save_root = "/mnt/tidalfs-bdsz01/usr/panlicheng/workspace/ProjDF-Meta/stats_ML3"
# dft_show.to_latex(f"{save_root}/long_term.tex")



dst_order = ['ETTm1', 'ETTm2', 'ETTh1', 'ETTh2', 'ECL', 'Weather', 'PEMS03', 'PEMS08']
model_order = ['MetaDF', 'TQNet', 'PDF', 'Fredformer', 'iTransformer', 'FreTS', 'TimesNet', 'MICN', 'TiDE', 'PatchTST', 'DLinear']
# 初始化统计字典：每个模型的MSE第1次数和MAE第1次数
mse_count = {model: 0 for model in model_order}
mae_count = {model: 0 for model in model_order}


contents = []
stats_contents = []  # 存储统计行内容
for dst in dst_order:
    dfi = dft[dft.data_id == dst]

    dfi['mse_rank'] = dfi.groupby('pred_len')['mse'].rank(ascending=True, method='dense')
    dfi['mae_rank'] = dfi.groupby('pred_len')['mae'].rank(ascending=True, method='dense')

    # MSE第1名：mse_rank=1的模型
    mse_first_models = dfi[dfi['mse_rank'] == 1]['model']
    for model in mse_first_models:
        if model in mse_count:  # 确保模型在model_order中
            mse_count[model] += 1
    
    # MAE第1名：mae_rank=1的模型
    mae_first_models = dfi[dfi['mae_rank'] == 1]['model']
    for model in mae_first_models:
        if model in mae_count:
            mae_count[model] += 1

    for metric in ['mse', 'mae']:
        dfi[f'{metric}_3f'] = dfi[metric].apply(lambda x: "{:.3f}".format(x))
        # 按排名标记：第1名→\bst{}，第2名→\subbst{}，其他→原格式
        conditions = [dfi[f'{metric}_rank'] == 1, dfi[f'{metric}_rank'] == 2]
        choices = [r'\bst{' + dfi[f'{metric}_3f'] + '}', r'\subbst{' + dfi[f'{metric}_3f'] + '}']
        dfi[f'{metric}_processed'] = np.select(conditions, choices, default=dfi[f'{metric}_3f'])
    dfi = dfi[['model', 'pred_len', 'mse_processed', 'mae_processed']]
    dfi.columns = ['model', 'pred_len', 'mse', 'mae']
    dfi['mse'] = dfi['mse'].apply(lambda x: r"\scalea{" + x + "}")  # 应用\scalea
    dfi['mae'] = dfi['mae'].apply(lambda x: r"\scalea{" + x + "}")
    for i, (pl, group) in enumerate(dfi.groupby('pred_len')):
        if pl != 'Avg':
            continue
        line = r"\multicolumn{2}{l}{\scalea{" + dst + r"}}" + '\n'
        for j, row in enumerate(group.itertuples(index=False)):
            line += f"& {row.mse} & {row.mae}"
        line += " \\\\"
        contents.append(line)
    if dst != dst_order[-1]:
        contents.append("\\midrule")


with open(f"{save_root}/long_term_avg.tex", "w") as f:
    f.write("\n".join(contents))
print("\n".join(contents))

\multicolumn{2}{l}{\scalea{ETTm1}}
& \scalea{\bst{0.371}} & \scalea{\bst{0.389}}& \scalea{\subbst{0.376}} & \scalea{\subbst{0.391}}& \scalea{0.387} & \scalea{0.396}& \scalea{0.387} & \scalea{0.398}& \scalea{0.411} & \scalea{0.414}& \scalea{0.414} & \scalea{0.421}& \scalea{0.438} & \scalea{0.430}& \scalea{0.396} & \scalea{0.421}& \scalea{0.413} & \scalea{0.407}& \scalea{0.389} & \scalea{0.400}& \scalea{0.403} & \scalea{0.407} \\
\midrule
\multicolumn{2}{l}{\scalea{ETTm2}}
& \scalea{\bst{0.270}} & \scalea{\bst{0.317}}& \scalea{\subbst{0.277}} & \scalea{\subbst{0.321}}& \scalea{0.283} & \scalea{0.331}& \scalea{0.280} & \scalea{0.324}& \scalea{0.295} & \scalea{0.336}& \scalea{0.316} & \scalea{0.365}& \scalea{0.302} & \scalea{0.334}& \scalea{0.308} & \scalea{0.364}& \scalea{0.286} & \scalea{0.328}& \scalea{0.303} & \scalea{0.344}& \scalea{0.342} & \scalea{0.392} \\
\midrule
\multicolumn{2}{l}{\scalea{ETTh1}}
& \scalea{\bst{0.431}} & \scalea{\bst{0.431}}& \scalea{0.449} & \scalea{0.439}& \sc

/tmp/ipykernel_253782/3944324592.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dfi['mse_rank'] = dfi.groupby('pred_len')['mse'].rank(ascending=True, method='dense')
/tmp/ipykernel_253782/3944324592.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dfi['mae_rank'] = dfi.groupby('pred_len')['mae'].rank(ascending=True, method='dense')
/tmp/ipykernel_253782/3944324592.py:34: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_index

## write avg

In [ ]:
save_root = "/mnt/tidalfs-bdsz01/usr/panlicheng/workspace/ProjDF-Meta/stats_ML3"
# dft_show.to_latex(f"{save_root}/long_term.tex")



dst_order = ['ETTm1', 'ETTm2', 'ETTh1', 'ETTh2', 'ECL', 'Weather', 'PEMS03', 'PEMS08']
model_order = ['MetaDF', 'TQNet', 'PDF', 'Fredformer', 'iTransformer', 'FreTS', 'TimesNet', 'MICN', 'TiDE', 'PatchTST', 'DLinear']
# 初始化统计字典：每个模型的MSE第1次数和MAE第1次数
mse_count = {model: 0 for model in model_order}
mae_count = {model: 0 for model in model_order}


contents = []
stats_contents = []  # 存储统计行内容
for dst in dst_order:
    dfi = dft[dft.data_id == dst]

    dfi['mse_rank'] = dfi.groupby('pred_len')['mse'].rank(ascending=True, method='dense')
    dfi['mae_rank'] = dfi.groupby('pred_len')['mae'].rank(ascending=True, method='dense')

    # MSE第1名：mse_rank=1的模型
    mse_first_models = dfi[dfi['mse_rank'] == 1]['model']
    for model in mse_first_models:
        if model in mse_count:  # 确保模型在model_order中
            mse_count[model] += 1
    
    # MAE第1名：mae_rank=1的模型
    mae_first_models = dfi[dfi['mae_rank'] == 1]['model']
    for model in mae_first_models:
        if model in mae_count:
            mae_count[model] += 1

    for metric in ['mse', 'mae']:
        dfi[f'{metric}_3f'] = dfi[metric].apply(lambda x: "{:.3f}".format(x))
        # 按排名标记：第1名→\bst{}，第2名→\subbst{}，其他→原格式
        conditions = [dfi[f'{metric}_rank'] == 1, dfi[f'{metric}_rank'] == 2]
        choices = [r'\bst{' + dfi[f'{metric}_3f'] + '}', r'\subbst{' + dfi[f'{metric}_3f'] + '}']
        dfi[f'{metric}_processed'] = np.select(conditions, choices, default=dfi[f'{metric}_3f'])
    dfi = dfi[['model', 'pred_len', 'mse_processed', 'mae_processed']]
    dfi.columns = ['model', 'pred_len', 'mse', 'mae']
    dfi['mse'] = dfi['mse'].apply(lambda x: r"\scalea{" + x + "}")  # 应用\scalea
    dfi['mae'] = dfi['mae'].apply(lambda x: r"\scalea{" + x + "}")
    for i, (pl, group) in enumerate(dfi.groupby('pred_len')):
        if pl != 'Avg':
            continue
        line = r"\multicolumn{2}{l}{\scalea{" + dst + r"}}" + '\n'
        for j, row in enumerate(group.itertuples(index=False)):
            line += f"& {row.mse} & {row.mae}"
        line += " \\\\"
        contents.append(line)
    if dst != dst_order[-1]:
        contents.append("\\midrule")


with open(f"{save_root}/long_term_avg.tex", "w") as f:
    f.write("\n".join(contents))
print("\n".join(contents))

\multicolumn{2}{l}{\scalea{ETTm1}}
& \scalea{\bst{0.371}} & \scalea{\bst{0.389}}& \scalea{\subbst{0.376}} & \scalea{\subbst{0.391}}& \scalea{0.387} & \scalea{0.396}& \scalea{0.387} & \scalea{0.398}& \scalea{0.411} & \scalea{0.414}& \scalea{0.414} & \scalea{0.421}& \scalea{0.438} & \scalea{0.430}& \scalea{0.396} & \scalea{0.421}& \scalea{0.413} & \scalea{0.407}& \scalea{0.389} & \scalea{0.400}& \scalea{0.403} & \scalea{0.407} \\
\midrule
\multicolumn{2}{l}{\scalea{ETTm2}}
& \scalea{\bst{0.270}} & \scalea{\bst{0.317}}& \scalea{\subbst{0.277}} & \scalea{\subbst{0.321}}& \scalea{0.283} & \scalea{0.331}& \scalea{0.280} & \scalea{0.324}& \scalea{0.295} & \scalea{0.336}& \scalea{0.316} & \scalea{0.365}& \scalea{0.302} & \scalea{0.334}& \scalea{0.308} & \scalea{0.364}& \scalea{0.286} & \scalea{0.328}& \scalea{0.303} & \scalea{0.344}& \scalea{0.342} & \scalea{0.392} \\
\midrule
\multicolumn{2}{l}{\scalea{ETTh1}}
& \scalea{\bst{0.431}} & \scalea{\bst{0.431}}& \scalea{0.449} & \scalea{0.439}& \sc

/tmp/ipykernel_253782/3944324592.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dfi['mse_rank'] = dfi.groupby('pred_len')['mse'].rank(ascending=True, method='dense')
/tmp/ipykernel_253782/3944324592.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dfi['mae_rank'] = dfi.groupby('pred_len')['mae'].rank(ascending=True, method='dense')
/tmp/ipykernel_253782/3944324592.py:34: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_index